In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a GPU program that converts an RGB image to grayscale on the GPU.
  Given an input RGB image represented as a 1D array of 32-bit floating point values,
  compute the corresponding grayscale image using the standard RGB to grayscale conversion formula.
</p>

<p>
  The conversion formula is: <code>gray = 0.299 × R + 0.587 × G + 0.114 × B</code>
</p>

<p>
  The input array <code>input</code> contains <code>height × width × 3</code> elements,
  where the RGB values for each pixel are stored consecutively (R, G, B, R, G, B, ...).
  The output array <code>output</code> should contain <code>height × width</code> grayscale values.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the array <code>output</code></li>
  <li>Use the exact coefficients: 0.299 for red, 0.587 for green, 0.114 for blue</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:  input = [255.0, 0.0, 0.0, 0.0, 255.0, 0.0, 0.0, 0.0, 255.0, 128.0, 128.0, 128.0], width=2, height=2
Output: output = [76.245, 149.685, 29.07, 128.0]
</pre>

<h2>Example 2:</h2>
<pre>
Input:  input = [100.0, 150.0, 200.0], width=1, height=1
Output: output = [140.75]
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 ≤ <code>width</code> ≤ 4096</li>
  <li>1 ≤ <code>height</code> ≤ 4096</li>
  <li><code>width × height</code> ≤ 4,194,304</li>
  <li>All RGB values are in the range [0.0, 255.0]</li>

  <li>Performance is measured with <code>height</code> = 2,048, <code>width</code> = 2,048</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

__global__ void rgb_to_grayscale_kernel(const float* input, float* output, int width, int height) {}

// input, output are device pointers
extern "C" void solve(const float* input, float* output, int width, int height) {
    int total_pixels = width * height;
    int threadsPerBlock = 256;
    int blocksPerGrid = (total_pixels + threadsPerBlock - 1) / threadsPerBlock;

    rgb_to_grayscale_kernel<<<blocksPerGrid, threadsPerBlock>>>(input, output, width, height);
    cudaDeviceSynchronize();
}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# input, output are tensors on the GPU
@cute.jit
def solve(input: cute.Tensor, output: cute.Tensor, width: cute.Int32, height: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# input is a tensor on GPU
@jax.jit
def solve(input: jax.Array, width: int, height: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


def rgb_to_grayscale_kernel(
    input: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    width: Int32,
    height: Int32,
):
    pass


# input, output are device pointers
@export
def solve(
    input: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    width: Int32,
    height: Int32,
) raises:
    var total_pixels = width * height
    var BLOCK_SIZE: Int32 = 256
    var ctx = DeviceContext()
    var num_blocks = ceildiv(total_pixels, BLOCK_SIZE)

    var _kernel = ctx.compile_function[rgb_to_grayscale_kernel, rgb_to_grayscale_kernel]()
    ctx.enqueue_function(
        _kernel, input, output, width, height, grid_dim=num_blocks, block_dim=BLOCK_SIZE
    )

    ctx.synchronize()


# Torch

In [ ]:
%%writefile solution.py
import torch


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, width: int, height: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


@triton.jit
def rgb_to_grayscale_kernel(input, output, width, height, BLOCK_SIZE: tl.constexpr):
    pass


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, width: int, height: int):
    total_pixels = width * height
    BLOCK_SIZE = 1024
    grid = (triton.cdiv(total_pixels, BLOCK_SIZE),)
    rgb_to_grayscale_kernel[grid](input, output, width, height, BLOCK_SIZE)


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="RGB to Grayscale", atol=1e-05, rtol=1e-05, num_gpus=1, access_tier="free"
        )

    def reference_impl(self, input: torch.Tensor, output: torch.Tensor, width: int, height: int):
        assert input.shape == (height * width * 3,)
        assert output.shape == (height * width,)
        assert input.dtype == output.dtype == torch.float32
        assert input.device == output.device

        # Reshape input to (height, width, 3) for easier processing
        rgb_image = input.view(height, width, 3)

        # Apply RGB to grayscale conversion: gray = 0.299*R + 0.587*G + 0.114*B
        grayscale = (
            0.299 * rgb_image[:, :, 0] + 0.587 * rgb_image[:, :, 1] + 0.114 * rgb_image[:, :, 2]
        )

        # Flatten and store in output
        output.copy_(grayscale.flatten())

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "input": (ctypes.POINTER(ctypes.c_float), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "width": (ctypes.c_int, "in"),
            "height": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        width, height = 2, 2
        # RGB values for a 2x2 image
        # Pixel (0,0): R=255, G=0, B=0 (red)
        # Pixel (0,1): R=0, G=255, B=0 (green)
        # Pixel (1,0): R=0, G=0, B=255 (blue)
        # Pixel (1,1): R=128, G=128, B=128 (gray)
        input_data = torch.tensor(
            [
                255.0,
                0.0,
                0.0,  # red
                0.0,
                255.0,
                0.0,  # green
                0.0,
                0.0,
                255.0,  # blue
                128.0,
                128.0,
                128.0,  # gray
            ],
            device="cuda",
            dtype=torch.float32,
        )
        output = torch.zeros(width * height, device="cuda", dtype=torch.float32)
        return {
            "input": input_data,
            "output": output,
            "width": width,
            "height": height,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        test_cases = []

        # Small test cases
        test_cases.append(
            {
                "input": torch.tensor(
                    [255.0, 0.0, 0.0], device="cuda", dtype=torch.float32
                ),  # red pixel
                "output": torch.zeros(1, device="cuda", dtype=torch.float32),
                "width": 1,
                "height": 1,
            }
        )

        test_cases.append(
            {
                "input": torch.tensor(
                    [0.0, 255.0, 0.0], device="cuda", dtype=torch.float32
                ),  # green pixel
                "output": torch.zeros(1, device="cuda", dtype=torch.float32),
                "width": 1,
                "height": 1,
            }
        )

        test_cases.append(
            {
                "input": torch.tensor(
                    [0.0, 0.0, 255.0], device="cuda", dtype=torch.float32
                ),  # blue pixel
                "output": torch.zeros(1, device="cuda", dtype=torch.float32),
                "width": 1,
                "height": 1,
            }
        )

        # 2x2 test case
        test_cases.append(
            {
                "input": torch.tensor(
                    [
                        100.0,
                        150.0,
                        200.0,  # mixed color 1
                        50.0,
                        75.0,
                        100.0,  # mixed color 2
                        200.0,
                        100.0,
                        50.0,  # mixed color 3
                        75.0,
                        125.0,
                        175.0,  # mixed color 4
                    ],
                    device="cuda",
                    dtype=torch.float32,
                ),
                "output": torch.zeros(4, device="cuda", dtype=torch.float32),
                "width": 2,
                "height": 2,
            }
        )

        # Edge cases: zeros and max values
        test_cases.append(
            {
                "input": torch.zeros(3, device="cuda", dtype=torch.float32),
                "output": torch.zeros(1, device="cuda", dtype=torch.float32),
                "width": 1,
                "height": 1,
            }
        )

        test_cases.append(
            {
                "input": torch.full((3,), 255.0, device="cuda", dtype=torch.float32),
                "output": torch.zeros(1, device="cuda", dtype=torch.float32),
                "width": 1,
                "height": 1,
            }
        )

        # Larger test cases
        for size in [4, 8, 16, 32]:
            input_size = size * size * 3
            test_cases.append(
                {
                    "input": torch.randint(
                        0, 256, (input_size,), device="cuda", dtype=torch.float32
                    ),
                    "output": torch.zeros(size * size, device="cuda", dtype=torch.float32),
                    "width": size,
                    "height": size,
                }
            )

        # Larger realistic sizes
        for w, h in [(100, 100), (64, 48)]:
            test_cases.append(
                {
                    "input": torch.empty(h * w * 3, device="cuda", dtype=torch.float32).uniform_(
                        0.0, 255.0
                    ),
                    "output": torch.zeros(h * w, device="cuda", dtype=torch.float32),
                    "width": w,
                    "height": h,
                }
            )

        # Non-square images
        test_cases.append(
            {
                "input": torch.randint(
                    0, 256, (2 * 3 * 3,), device="cuda", dtype=torch.float32
                ),  # 2x3 image
                "output": torch.zeros(2 * 3, device="cuda", dtype=torch.float32),
                "width": 3,
                "height": 2,
            }
        )

        test_cases.append(
            {
                "input": torch.randint(
                    0, 256, (3 * 2 * 3,), device="cuda", dtype=torch.float32
                ),  # 3x2 image
                "output": torch.zeros(3 * 2, device="cuda", dtype=torch.float32),
                "width": 2,
                "height": 3,
            }
        )

        return test_cases

    def generate_performance_test(self) -> Dict[str, Any]:
        width, height = 2048, 2048
        input_size = width * height * 3
        output_size = width * height
        return {
            "input": torch.randint(0, 256, (input_size,), device="cuda", dtype=torch.float32),
            "output": torch.zeros(output_size, device="cuda", dtype=torch.float32),
            "width": width,
            "height": height,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
